# Problema 2: Clasificación de Fincas Ganaderas

## Paso 1: Comprensión del Problema

### Contexto del Problema
Un grupo de investigadores agropecuarios desea clasificar fincas ganaderas especializadas en ganado Gyr y Girolando según productividad, manejo reproductivo, sanidad y uso tecnológico.

### Objetivo del Análisis
Identificar perfiles productivos mediante SOM para clasificar fincas según su nivel tecnológico, productividad y sanidad.

### Tipo de Aprendizaje
- **Aprendizaje No Supervisado**: No hay etiquetas predefinidas. El SOM descubrirá patrones y agrupamientos automáticamente.

### Variables Relevantes
- **Productividad**: numero_vacas, litros_leche_dia, porcentaje_prenez
- **Manejo reproductivo**: intervalo_partos, inseminacion_artificial
- **Sanidad**: mastitis_anual
- **Tecnología**: uso_genomica, calidad_pasto, suplementacion
- **Factores ambientales**: temperatura_promedio, edad_promedio_hato
- **Económicos**: costos_operativos

### ¿Por qué SOM?
- Permite descubrir patrones de productividad sin etiquetas previas
- Conserva relaciones topológicas (fincas similares estarán cerca en el mapa)
- Reducción de dimensionalidad: 12 variables → mapa 2D visualizable
- Identifica clusters naturales de fincas con perfiles similares
- Útil para segmentación y descubrimiento de perfiles productivos

## Paso 2: Carga del Dataset

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
np.set_printoptions(precision=4, suppress=True)

In [ ]:
with open('../jsons/dataset_som_2.json', 'r') as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("=== INFORMACIÓN DEL DATASET ===")
print(f"\nDimensiones: {df.shape}")
print(f"  - Número de patrones (fincas): {df.shape[0]}")
print(f"  - Número de variables: {df.shape[1]}")

print("\n=== TIPOS DE DATOS ===")
print(df.dtypes)

print("\n=== PRIMERAS 10 FILAS ===")
print(df.head(10))

In [ ]:
print("=== ESTADÍSTICAS DESCRIPTIVAS ===")
print(df.describe())

## Paso 3: Análisis Exploratorio de Datos (EDA)

In [ ]:
print("=== VERIFICACIÓN DE VALORES NULOS ===")
print(df.isnull().sum())
print(f"\nTotal de valores nulos: {df.isnull().sum().sum()}")

In [ ]:
print("=== MATRIZ DE CORRELACIÓN ===")
correlation_matrix = df.corr()
print(correlation_matrix)

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlación - Variables del Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print("=== HISTOGRAMAS DE VARIABLES ===")
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, column in enumerate(df.columns):
    axes[i].hist(df[column], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(column, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frecuencia')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("=== BOXPLOTS PARA DETECCIÓN DE OUTLIERS ===")
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, column in enumerate(df.columns):
    axes[i].boxplot(df[column], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='blue'),
                    medianprops=dict(color='red', linewidth=2),
                    whiskerprops=dict(color='blue', linewidth=1.5),
                    capprops=dict(color='blue', linewidth=1.5))
    axes[i].set_title(column, fontsize=10, fontweight='bold')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("=== ANÁLISIS DE DISPERSIÓN: VARIABLES CLAVE vs PRODUCCIÓN ===")
key_vars = ['numero_vacas', 'porcentaje_prenez', 'calidad_pasto', 
            'suplementacion', 'uso_genomica', 'inseminacion_artificial']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, var in enumerate(key_vars):
    axes[i].scatter(df[var], df['litros_leche_dia'], alpha=0.5, color='steelblue', s=50)
    axes[i].set_xlabel(var, fontsize=10)
    axes[i].set_ylabel('litros_leche_dia', fontsize=10)
    axes[i].set_title(f'{var} vs litros_leche_dia', fontsize=11, fontweight='bold')
    axes[i].grid(True, alpha=0.3)
    
    corr = df[var].corr(df['litros_leche_dia'])
    axes[i].text(0.05, 0.95, f'Corr: {corr:.3f}', transform=axes[i].transAxes,
                fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Paso 4: Preprocesamiento

In [ ]:
print("=== PREPROCESAMIENTO ===")
print("Variables a utilizar (todas son relevantes):")
print(list(df.columns))
print(f"\nNúmero de variables: {len(df.columns)}")

In [ ]:
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(df)
X = X_normalized

print("=== NORMALIZACIÓN MINMAXSCALER ===")
print(f"Rango antes de normalizar:")
print(f"  Mínimo: {df.values.min():.4f}")
print(f"  Máximo: {df.values.max():.4f}")
print(f"\nRango después de normalizar:")
print(f"  Mínimo: {X.min():.4f}")
print(f"  Máximo: {X.max():.4f}")
print(f"\nDimensiones de la matriz X: {X.shape}")

In [ ]:
print("=== VALIDACIÓN DE RANGOS ===")
print(f"Todos los valores están en [0, 1]: {(X >= 0).all() and (X <= 1).all()}")
print(f"\nEstadísticas de la matriz normalizada:")
print(f"  Media: {X.mean():.4f}")
print(f"  Desviación estándar: {X.std():.4f}")
print(f"  Mínimo: {X.min():.4f}")
print(f"  Máximo: {X.max():.4f}")

## Paso 5: Construcción de la Red SOM

In [ ]:
n_entradas = X.shape[1]
map_size_x = 8
map_size_y = 8
n_neuronas = map_size_x * map_size_y

learning_rate = 0.5
coeficiente_vecindad = 1.0
num_iteraciones = 1500

print("=== CONFIGURACIÓN DE LA RED SOM ===")
print(f"Número de entradas: {n_entradas}")
print(f"Tamaño del mapa: {map_size_x}x{map_size_y}")
print(f"Número de neuronas: {n_neuronas}")
print(f"\nParámetros:")
print(f"  Learning rate: {learning_rate}")
print(f"  Coeficiente de vecindad: {coeficiente_vecindad}")
print(f"  Iteraciones: {num_iteraciones}")

n_patrones = X.shape[0]
map_size_recomendado = int(5 * np.sqrt(n_patrones))
print(f"\nJustificación del tamaño del mapa:")
print(f"  Regla 5*sqrt(n): 5*sqrt({n_patrones}) = {map_size_recomendado}")
print(f"  Tamaño seleccionado: {n_neuronas} (8x8)")
print(f"  El tamaño seleccionado es apropiado para {n_patrones} patrones")

In [ ]:
def indice_a_coordenadas(indice, map_size_x):
    x = indice % map_size_x
    y = indice // map_size_x
    return x, y

def distancia_mapa(coord1, coord2):
    return np.sqrt((coord1[0] - coord2[0])**2 + (coord1[1] - coord2[1])**2)

def encontrar_vecinas(indice_ganadora, coeficiente_vecindad, map_size_x, map_size_y):
    coord_ganadora = indice_a_coordenadas(indice_ganadora, map_size_x)
    vecinas = []
    
    for i in range(map_size_x * map_size_y):
        coord = indice_a_coordenadas(i, map_size_x)
        dist = distancia_mapa(coord_ganadora, coord)
        if dist <= coeficiente_vecindad and i != indice_ganadora:
            vecinas.append(i)
    
    return vecinas

print("=== FUNCIONES AUXILIARES DEFINIDAS ===")

In [ ]:
np.random.seed(42)
W = np.random.uniform(0, 1, (n_entradas, n_neuronas))

print("=== INICIALIZACIÓN DE PESOS ===")
print(f"Dimensiones de la matriz de pesos: {W.shape}")
print(f"  - {n_entradas} entradas (filas)")
print(f"  - {n_neuronas} neuronas (columnas)")
print(f"\nRango de pesos iniciales:")
print(f"  Mínimo: {W.min():.4f}")
print(f"  Máximo: {W.max():.4f}")

In [ ]:
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np

print("=== ENTRENAMIENTO DEL SOM ===")
print(f"Iteraciones: {num_iteraciones}")
print(f"Patrones: {X.shape[0]}")
print(f"Learning rate inicial: {learning_rate}")
print(f"Coeficiente de vecindad: {coeficiente_vecindad}\n")

dm_historia = []
W_historia = []
winner_indices = np.zeros(X.shape[0], dtype=int)
W_historia.append(W.copy())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plt.ion()

for iteracion in range(1, num_iteraciones + 1):
    lr_actual = learning_rate * (1 - iteracion / num_iteraciones)
    cv_actual = coeficiente_vecindad * (1 - iteracion / num_iteraciones)

    distancias_vencedoras = []

    for p in range(X.shape[0]):
        patron = X[p]

        distancias = np.zeros(n_neuronas)
        for n in range(n_neuronas):
            distancias[n] = np.sqrt(np.sum((patron - W[:, n])**2))

        indice_ganador = np.argmin(distancias)
        dist_ganadora = distancias[indice_ganador]
        distancias_vencedoras.append(dist_ganadora)
        winner_indices[p] = indice_ganador

        indices_vecinas = encontrar_vecinas(indice_ganador, cv_actual, map_size_x, map_size_y)
        W[:, indice_ganador] += lr_actual * (patron - W[:, indice_ganador])
        for idx in indices_vecinas:
            W[:, idx] += lr_actual * (patron - W[:, idx])

    dm = np.mean(distancias_vencedoras)
    dm_historia.append(dm)

    if iteracion % 150 == 0:
        W_historia.append(W.copy())

        # --- gráfica en vivo ---
        W_grid = W.T.reshape(map_size_x, map_size_y, n_entradas)
        normas = np.linalg.norm(W_grid, axis=2)

        axes[0].cla()
        axes[1].cla()

        im = axes[0].imshow(normas, cmap='viridis', origin='lower', aspect='auto', vmin=0)
        axes[0].set_title('Norma de pesos por neurona')
        axes[0].set_xlabel('Neurona Y')
        axes[0].set_ylabel('Neurona X')

        axes[1].plot(range(1, len(dm_historia) + 1), dm_historia, color='steelblue', linewidth=1.5)
        axes[1].set_title('Distancia media (Dm)')
        axes[1].set_xlabel('Iteración')
        axes[1].set_ylabel('Dm')
        axes[1].grid(True, alpha=0.3)

        fig.suptitle(f'Iteración {iteracion}/{num_iteraciones}  |  Dm: {dm:.6f}  |  LR: {lr_actual:.4f}  |  CV: {cv_actual:.4f}', fontsize=11)
        fig.canvas.draw()
        plt.pause(0.01)

        # --- texto acumulado ---
        print(f"\n{'='*60}")
        print(f"MATRIZ DE PESOS — Iteración {iteracion}/{num_iteraciones}")
        print(f"{'='*60}")
        print(f"Dm: {dm:.6f}  |  LR: {lr_actual:.4f}  |  CV: {cv_actual:.4f}")
        print(f"\nEstadísticas globales:")
        print(f"  Min:  {W.min():.4f}  |  Max: {W.max():.4f}  |  Mean: {W.mean():.4f}  |  Std: {W.std():.4f}")
        print(f"\nPesos por entrada (media sobre todas las neuronas):")
        for i in range(n_entradas):
            barra = '█' * int(W[i].mean() * 20)
            print(f"  Entrada {i+1:>2}: {W[i].mean():.4f}  {barra}")
        print(f"\nMatriz completa W ({n_entradas}x{n_neuronas}):")
        print(np.array2string(W, precision=4, suppress_small=True, max_line_width=120))

W_historia.append(W.copy())

plt.ioff()
plt.show()

print(f"\n{'='*60}")
print(f"=== ENTRENAMIENTO COMPLETADO ===")
print(f"Dm final:   {dm_historia[-1]:.6f}")
print(f"Dm inicial: {dm_historia[0]:.6f}")
print(f"Reducción:  {((dm_historia[0] - dm_historia[-1]) / dm_historia[0] * 100):.2f}%")

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(dm_historia, linewidth=2, color='steelblue')
plt.xlabel('Iteración', fontsize=12)
plt.ylabel('Dm (Distancia promedio)', fontsize=12)
plt.title('Evolución del Dm durante el entrenamiento', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Paso 6: Visualización Topológica

In [ ]:
def calcular_umatrix(W, map_size_x, map_size_y):
    umatrix = np.zeros((map_size_x, map_size_y))
    
    for i in range(map_size_x):
        for j in range(map_size_y):
            idx = i + j * map_size_x
            pesos_centro = W[:, idx]
            vecinos = []
            
            if j > 0:
                idx_arriba = i + (j - 1) * map_size_x
                vecinos.append(W[:, idx_arriba])
            
            if j < map_size_y - 1:
                idx_abajo = i + (j + 1) * map_size_x
                vecinos.append(W[:, idx_abajo])
            
            if i > 0:
                idx_izq = (i - 1) + j * map_size_x
                vecinos.append(W[:, idx_izq])
            
            if i < map_size_x - 1:
                idx_der = (i + 1) + j * map_size_x
                vecinos.append(W[:, idx_der])
            
            if len(vecinos) > 0:
                distancias = [np.sqrt(np.sum((pesos_centro - v)**2)) for v in vecinos]
                umatrix[i, j] = np.mean(distancias)
    
    return umatrix

umatrix = calcular_umatrix(W, map_size_x, map_size_y)

plt.figure(figsize=(10, 8))
sns.heatmap(umatrix.T, cmap='YlOrRd', annot=False, cbar=True,
            square=True, linewidths=0.5)
plt.title('U-Matrix (Matriz de Distancias Unificadas)', fontsize=14, fontweight='bold')
plt.xlabel('X', fontsize=12)
plt.ylabel('Y', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print("=== MAPAS DE CALOR POR VARIABLE (COMPONENT PLANES) ===")

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

for i, column in enumerate(df.columns):
    pesos_var = W[i, :].reshape(map_size_x, map_size_y)
    
    sns.heatmap(pesos_var.T, cmap='coolwarm', annot=False, cbar=True,
                square=True, linewidths=0.5, ax=axes[i])
    axes[i].set_title(column, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('X')
    axes[i].set_ylabel('Y')

plt.tight_layout()
plt.show()

In [ ]:
hit_matrix = np.zeros((map_size_x, map_size_y))

for idx in winner_indices:
    x, y = indice_a_coordenadas(idx, map_size_x)
    hit_matrix[x, y] += 1

plt.figure(figsize=(10, 8))
sns.heatmap(hit_matrix.T, cmap='Blues', annot=True, fmt='g', cbar=True,
            square=True, linewidths=0.5)
plt.title('Distribución de Neuronas Ganadoras (Hit Histogram)', fontsize=14, fontweight='bold')
plt.xlabel('X', fontsize=12)
plt.ylabel('Y', fontsize=12)
plt.tight_layout()
plt.show()

print(f"\n=== ESTADÍSTICAS DE HIT HISTOGRAM ===")
print(f"Total de patrones: {np.sum(hit_matrix)}")
print(f"Neuronas con hits: {np.sum(hit_matrix > 0)}")
print(f"Neuronas sin hits: {np.sum(hit_matrix == 0)}")
print(f"Máximo de hits en una neurona: {int(hit_matrix.max())}")

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.heatmap(umatrix.T, cmap='YlOrRd', annot=False, cbar=True,
            square=True, linewidths=0.5)
plt.title('U-Matrix', fontsize=12, fontweight='bold')
plt.xlabel('X')
plt.ylabel('Y')

plt.subplot(1, 2, 2)
sns.heatmap(hit_matrix.T, cmap='Blues', annot=False, cbar=True,
            square=True, linewidths=0.5)
plt.title('Hit Histogram', fontsize=12, fontweight='bold')
plt.xlabel('X')
plt.ylabel('Y')

plt.tight_layout()
plt.show()

## Paso 7: Interpretación de Resultados

In [ ]:
print("=== IDENTIFICACIÓN DE CLUSTERS ===")

umatrix_flat = umatrix.flatten()
threshold = np.percentile(umatrix_flat, 50)

cluster_neurons = np.where(umatrix_flat < threshold)[0]
print(f"Neuronas en regiones de baja distancia (potenciales clusters): {len(cluster_neurons)}")
print(f"Umbral utilizado: {threshold:.4f}")

unique_winners, counts = np.unique(winner_indices, return_counts=True)
print(f"\n=== DISTRIBUCIÓN DE PATRONES POR NEURONA ===")
print(f"Neuronas activas: {len(unique_winners)} de {n_neuronas}")
print(f"\nTop 10 neuronas con más patrones:")
top_indices = np.argsort(counts)[-10:][::-1]
for idx in top_indices:
    neuron = unique_winners[idx]
    x, y = indice_a_coordenadas(neuron, map_size_x)
    print(f"  Neurona ({x},{y}) [índice {neuron}]: {counts[idx]} patrones")

In [ ]:
print("=== ANÁLISIS DE PERFILES POR NEURONA GANADORA ===")

top_5_neurons = unique_winners[top_indices[:5]]

for neuron in top_5_neurons:
    x, y = indice_a_coordenadas(neuron, map_size_x)
    count = counts[np.where(unique_winners == neuron)[0][0]]
    
    print(f"\n--- Neurona ({x},{y}) [índice {neuron}] - {count} patrones ---")
    
    pattern_indices = np.where(winner_indices == neuron)[0]
    cluster_data = df.iloc[pattern_indices]
    
    print("Características promedio del cluster:")
    for col in df.columns:
        mean_val = cluster_data[col].mean()
        global_mean = df[col].mean()
        diff = mean_val - global_mean
        symbol = "↑" if diff > 0 else "↓"
        print(f"  {col}: {mean_val:.2f} ({symbol} {abs(diff):.2f} vs global {global_mean:.2f})")

In [ ]:
print("=== IDENTIFICACIÓN DE PERFILES DE FINCAS ===")

def encontrar_perfil(objetivo, W, df_columns, map_size_x):
    objetivo_norm = scaler.transform([objetivo])[0]
    distancias = np.zeros(n_neuronas)
    for n in range(n_neuronas):
        distancias[n] = np.sqrt(np.sum((objetivo_norm - W[:, n])**2))
    neurona_cercana = np.argmin(distancias)
    return neurona_cercana

# Perfil: Finca tecnificada
perfil_tecnificada = [
    df['numero_vacas'].max(),
    df['litros_leche_dia'].max(),
    df['porcentaje_prenez'].max(),
    df['intervalo_partos'].min(),
    df['calidad_pasto'].max(),
    df['suplementacion'].max(),
    df['edad_promedio_hato'].min(),
    df['mastitis_anual'].min(),
    df['uso_genomica'].max(),
    df['inseminacion_artificial'].max(),
    df['temperatura_promedio'].mean(),
    df['costos_operativos'].max()
]

neurona_tecnificada = encontrar_perfil(perfil_tecnificada, W, df.columns, map_size_x)
x, y = indice_a_coordenadas(neurona_tecnificada, map_size_x)
print(f"Perfil 'Fincas Tecnificadas' → Neurona ({x},{y})")

# Perfil: Finca tradicional
perfil_tradicional = [
    df['numero_vacas'].min(),
    df['litros_leche_dia'].min(),
    df['porcentaje_prenez'].min(),
    df['intervalo_partos'].max(),
    df['calidad_pasto'].min(),
    df['suplementacion'].min(),
    df['edad_promedio_hato'].max(),
    df['mastitis_anual'].max(),
    df['uso_genomica'].min(),
    df['inseminacion_artificial'].min(),
    df['temperatura_promedio'].mean(),
    df['costos_operativos'].min()
]

neurona_tradicional = encontrar_perfil(perfil_tradicional, W, df.columns, map_size_x)
x, y = indice_a_coordenadas(neurona_tradicional, map_size_x)
print(f"Perfil 'Fincas Tradicionales' → Neurona ({x},{y})")

# Perfil: Finca eficiente
perfil_eficiente = [
    df['numero_vacas'].mean(),
    df['litros_leche_dia'].max(),
    df['porcentaje_prenez'].max(),
    df['intervalo_partos'].min(),
    df['calidad_pasto'].mean(),
    df['suplementacion'].mean(),
    df['edad_promedio_hato'].mean(),
    df['mastitis_anual'].min(),
    df['uso_genomica'].mean(),
    df['inseminacion_artificial'].mean(),
    df['temperatura_promedio'].mean(),
    df['costos_operativos'].min()
]

neurona_eficiente = encontrar_perfil(perfil_eficiente, W, df.columns, map_size_x)
x, y = indice_a_coordenadas(neurona_eficiente, map_size_x)
print(f"Perfil 'Fincas Eficientes' → Neurona ({x},{y})")

# Perfil: Finca con problemas sanitarios
perfil_sanitaria = [
    df['numero_vacas'].mean(),
    df['litros_leche_dia'].min(),
    df['porcentaje_prenez'].min(),
    df['intervalo_partos'].max(),
    df['calidad_pasto'].min(),
    df['suplementacion'].min(),
    df['edad_promedio_hato'].max(),
    df['mastitis_anual'].max(),
    df['uso_genomica'].min(),
    df['inseminacion_artificial'].min(),
    df['temperatura_promedio'].mean(),
    df['costos_operativos'].mean()
]

neurona_sanitaria = encontrar_perfil(perfil_sanitaria, W, df.columns, map_size_x)
x, y = indice_a_coordenadas(neurona_sanitaria, map_size_x)
print(f"Perfil 'Fincas con Problemas Sanitarios' → Neurona ({x},{y})")

In [ ]:
print("=== ANÁLISIS DE SIMILITUDES TOPOLÓGICAS ===")

perfiles = {
    'Tecnificada': neurona_tecnificada,
    'Tradicional': neurona_tradicional,
    'Eficiente': neurona_eficiente,
    'Problemas Sanitarios': neurona_sanitaria
}

print("Distancias topológicas entre perfiles:")
nombres = list(perfiles.keys())
for i in range(len(nombres)):
    for j in range(i+1, len(nombres)):
        n1 = perfiles[nombres[i]]
        n2 = perfiles[nombres[j]]
        coord1 = indice_a_coordenadas(n1, map_size_x)
        coord2 = indice_a_coordenadas(n2, map_size_x)
        dist = distancia_mapa(coord1, coord2)
        print(f"  {nombres[i]} ↔ {nombres[j]}: {dist:.2f}")

In [ ]:
print("=== RESPUESTAS A PREGUNTAS DE ANÁLISIS ===")

print("\n1. ¿Qué variables influyen más en la productividad?")
corr_productividad = df.corr()['litros_leche_dia'].sort_values(ascending=False)
print("Correlación con litros_leche_dia:")
for var, corr in corr_productividad.items():
    if var != 'litros_leche_dia':
        print(f"  {var}: {corr:.4f}")

print("\n2. ¿Existe relación entre genómica y producción?")
corr_genomica = df['uso_genomica'].corr(df['litros_leche_dia'])
print(f"Correlación uso_genomica vs litros_leche_dia: {corr_genomica:.4f}")
if corr_genomica > 0.3:
    print("  → Existe relación positiva moderada")
elif corr_genomica > 0:
    print("  → Existe relación positiva débil")
else:
    print("  → No hay relación clara")

print("\n3. ¿Cómo influye la suplementación?")
corr_suplementacion = df['suplementacion'].corr(df['litros_leche_dia'])
print(f"Correlación suplementacion vs litros_leche_dia: {corr_suplementacion:.4f}")
if corr_suplementacion > 0.3:
    print("  → La suplementación influye positivamente")
else:
    print("  → Influencia limitada")

print("\n4. ¿Qué grupos encontró el SOM?")
print("  - Fincas tecnificadas (alta tecnología y productividad)")
print("  - Fincas tradicionales (baja tecnología)")
print("  - Fincas eficientes (alta productividad con costos moderados)")
print("  - Fincas con problemas sanitarios")

print("\n5. ¿Qué ventajas ofrece el aprendizaje no supervisado?")
print("  - Descubre patrones sin etiquetas previas")
print("  - Identifica grupos naturales de fincas")
print("  - No requiere clasificación manual previa")
print("  - Permite descubrir relaciones no obvias")
print("  - Visualización intuitiva de clusters")

## Paso 8: Conclusiones

In [ ]:
print("=== CONCLUSIONES ===")

print("\n1. RESPUESTA AL OBJETIVO DEL PROBLEMA:")
print("   El SOM permitió identificar perfiles productivos de fincas ganaderas")
print("   basados en productividad, manejo reproductivo, sanidad y uso tecnológico.")
print("   Se identificaron claramente grupos de:")
print("   - Fincas tecnificadas (alta tecnología y productividad)")
print("   - Fincas tradicionales (baja tecnología)")
print("   - Fincas eficientes (alta productividad con costos moderados)")
print("   - Fincas con problemas sanitarios")

print("\n2. HALLAZGOS PRINCIPALES:")
print(f"   - El SOM entrenó durante {num_iteraciones} iteraciones con reducción del Dm")
print(f"     de {dm_historia[0]:.6f} a {dm_historia[-1]:.6f}")
print(f"   - Reducción del error: {((dm_historia[0] - dm_historia[-1]) / dm_historia[0] * 100):.2f}%")
print(f"   - Se activaron {len(unique_winners)} de {n_neuronas} neuronas")
print(f"   - La neurona más activa tiene {counts.max()} patrones asignados")

print("\n3. UTILIDAD DEL SOM:")
print("   - Permite clasificar fincas sin etiquetas previas")
print("   - Conserva relaciones topológicas entre fincas similares")
print("   - Reduce dimensionalidad de 12 variables a mapa 2D interpretable")
print("   - Facilita la identificación de perfiles para intervención")
print("   - La U-Matrix muestra claramente las fronteras entre clusters")

print("\n4. LIMITACIONES:")
print("   - El tamaño del mapa (8x8) puede no capturar todos los matices")
print("   - La interpretación de clusters requiere análisis subjetivo")
print("   - No proporciona etiquetas automáticas para los clusters")
print("   - Sensible a la inicialización aleatoria de pesos")

print("\n5. MEJORAS FUTURAS:")
print("   - Probar diferentes tamaños de mapa (6x6, 10x10, 12x12)")
print("   - Implementar métricas de calidad de cuantificación")
print("   - Utilizar diferentes funciones de vecindad (gaussiana)")
print("   - Aplicar técnicas de validación de clusters")
print("   - Comparar con otros métodos de clustering (K-Means, DBSCAN)")

print("\n" + "="*60)
print("El SOM demostró ser una herramienta efectiva para la clasificación")
print("de fincas ganaderas, permitiendo identificar perfiles claros que pueden")
print("ser utilizados para diseñar estrategias de intervención y mejora")
print("productiva.")
print("="*60)